# 🦜🔗 LangChain 101: Comprehensive Guide with Local Ollama

Welcome to **LangChain 101**! This notebook provides a complete, hands-on tutorial covering all core concepts of **LangChain** using **local Ollama models** (`nemotron-3-nano:4b`, `gemma4:e2b`, and `nomic-embed-text:latest`).

### 🎯 Learning Objectives:
1. **Model Connection & Basics**: Invoking, streaming, and batching with `ChatOllama`.
2. **Prompts & Messages**: `ChatPromptTemplate`, message types, and `MessagesPlaceholder`.
3. **Output Parsers**: `StrOutputParser` and Pydantic-based `Structured Output`.
4. **LCEL (LangChain Expression Language)**: The pipe operator (`|`), `RunnableParallel`, `RunnablePassthrough`, and `RunnableLambda`.
5. **Memory & LangGraph Persistence**: Conversation persistence using LangGraph's recommended `MemorySaver` checkpointer.
6. **Tool Calling & Custom Tools**: Clear step-by-step breakdown of LLM tool requests vs. Python tool execution.
7. **Agent Construction**: Building interactive, tool-calling agents.
8. **RAG (Retrieval-Augmented Generation)**: Splitting documents, vector embeddings with `OllamaEmbeddings`, vector stores, and retriever chains.

--- 
## 0. Setup & Local Ollama Verification

First, let's set up imports and initialize our local Ollama model instance. We can use `nemotron-3-nano:4b` or `gemma4:e2b`.

In [5]:
import os
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.chat_models import init_chat_model

# Select local Ollama model
MODEL_NAME = "nemotron-3-nano:4b"  # Or "gemma4:e2b"
EMBED_MODEL = "nomic-embed-text:latest"

# llm = ChatOllama(
#     model=MODEL_NAME,
#     temperature=0.7
# )

# Initialize gemini chat model
llm = init_chat_model("google_genai:gemini-3.1-flash-lite")

print(f"✅ Connected to local Ollama model: {MODEL_NAME}")

✅ Connected to local Ollama model: nemotron-3-nano:4b


--- 
## 1. Model Invocation: `invoke`, `stream`, & `batch` 

LangChain models implement the standard **Runnable interface**, giving you uniform methods to interact with models:

In [6]:
# 1. Direct Invoke
response = llm.invoke("Give me a 1-sentence definition of Artificial Intelligence.")
print("--- Standard Response ---")
print(response.content)

# 2. Stream Response
print("\n--- Streaming Response ---")
for chunk in llm.stream("Count from 1 to 5 rapidly."):
    print(chunk.content, end="", flush=True)
print()

# 3. Batch Invocations
print("\n--- Batch Responses ---")
questions = [
    "What is Python?",
    "What is Ollama?"
]
batch_results = llm.batch(questions)
for q, res in zip(questions, batch_results):
    print(f"Q: {q}\nA: {res.content.strip()}\n")

--- Standard Response ---
[{'type': 'text', 'text': 'Artificial intelligence is a field of computer science dedicated to creating systems capable of performing tasks that typically require human intelligence, such as learning, reasoning, problem-solving, and perception.', 'extras': {'signature': 'EjQKMgERTTIPtup8PpuZPQIOwe0r1i8RKiSAS7LlDN7xgqFc8nD5bKORhgviI49P8KwEUM0N'}}]

--- Streaming Response ---
[{'type': 'text', 'text': '1, 2, 3, 4, 5.', 'index': 0}][{'type': 'text', 'text': '', 'extras': {'signature': 'EjQKMgERTTIPp3kYwhnDi3tuyIdgJtY4kaPYDc3/Ec3+z41Mg6sIPeKXuEbY/G8pD/fvsoDL'}, 'index': 0}][]

--- Batch Responses ---


AttributeError: 'list' object has no attribute 'strip'

--- 
## 2. Prompts & Messages: `ChatPromptTemplate`

In modern AI applications, structured chat messages (`SystemMessage`, `HumanMessage`, `AIMessage`) guide LLM behavior.
`ChatPromptTemplate` parameterizes system instructions and user inputs.

In [7]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Creating a ChatPromptTemplate with System & Human roles
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful programming tutor skilled in {language}. Keep explanations concise."),
    ("human", "Explain {concept} in 2 simple bullet points.")
])

# Formatting the prompt
formatted_messages = prompt_template.format_messages(language="Python", concept="Decorators")
print("Formatted Prompt Messages:", formatted_messages)

# Invoking the model with the formatted prompt
response = llm.invoke(formatted_messages)
print("\nModel Response:")
print(response.content)

Formatted Prompt Messages: [SystemMessage(content='You are a helpful programming tutor skilled in Python. Keep explanations concise.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain Decorators in 2 simple bullet points.', additional_kwargs={}, response_metadata={})]

Model Response:
[{'type': 'text', 'text': 'Here are decorators in two simple points:\n\n*   **Function Wrappers:** A decorator is a function that takes another function as an argument to extend or modify its behavior without changing its source code.\n*   **The `@` Syntax:** You apply them by placing the `@decorator_name` syntax directly above the definition of the target function.', 'extras': {'signature': 'EjQKMgERTTIP/JGwLZIrmhHhbn48dVKGfeNmDoZo2Nmkn/S43U87RbI0grZt0cfENRFqXs7G'}}]


--- 
## 3. Output Parsers & Pydantic Structured Output

Raw LLM outputs are `AIMessage` objects. Output parsers convert LLM outputs into clean strings, dictionaries, or typed Pydantic models.

In [9]:
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

# 1. StrOutputParser: Extracts string content from AIMessage
str_parser = StrOutputParser()
clean_text = str(str_parser.invoke(response))
print("Parsed String Content:", type(clean_text), f"'{clean_text[:50]}...'")

# 2. Structured Output with Pydantic
class ProgrammingConcept(BaseModel):
    name: str = Field(description="The name of the concept")
    language: str = Field(description="Programming language")
    summary: str = Field(description="1-sentence summary of the concept")
    difficulty: str = Field(description="Beginner, Intermediate, or Advanced")

# Bind Pydantic schema to model
structured_llm = llm.with_structured_output(ProgrammingConcept)

result = structured_llm.invoke("Explain Recursion in Python")
print("\nStructured Pydantic Output:")
print(f"Concept: {result.name} ({result.language})")
print(f"Difficulty: {result.difficulty}")
print(f"Summary: {result.summary}")

Parsed String Content: <class 'str'> 'Here are decorators in two simple points:

*   **F...'

Structured Pydantic Output:
Concept: Recursion (Python)
Difficulty: Intermediate
Summary: A programming technique where a function calls itself to solve smaller instances of the same problem until a base case is reached.


--- 
## 4. LangChain Expression Language (LCEL)

LCEL allows you to compose complex chains using the pipe operator `|`.
A standard chain follows: `Prompt | Model | OutputParser`.

You can also use:
- `RunnableParallel`: Run multiple steps in parallel.
- `RunnablePassthrough`: Pass inputs through unchanged.
- `RunnableLambda`: Wrap custom Python functions as runnable components.

In [10]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

# 1. Basic LCEL Chain
chain1 = prompt_template | llm | StrOutputParser()
output1 = chain1.invoke({"language": "Python", "concept": "List Comprehensions"})
print("--- LCEL Chain 1 Output ---")
print(output1)

# 2. Advanced LCEL: Parallel Composition
poem_prompt = ChatPromptTemplate.from_template("Write a 2-line poem about {topic}.")
joke_prompt = ChatPromptTemplate.from_template("Tell a 1-line joke about {topic}.")

poem_chain = poem_prompt | llm | StrOutputParser()
joke_chain = joke_prompt | llm | StrOutputParser()

# Combine in parallel
combined_chain = RunnableParallel(
    poem=poem_chain,
    joke=joke_chain
)

parallel_output = combined_chain.invoke({"topic": "Artificial Intelligence"})
print("\n--- Parallel Output ---")
print("POEM:\n", parallel_output['poem'])
print("JOKE:\n", parallel_output['joke'])

--- LCEL Chain 1 Output ---
Here is a concise explanation of list comprehensions:

*   **Compact Syntax:** They provide a one-line way to create a new list by applying an expression to each item in an existing sequence (like a list or range).
*   **Cleaner Code:** They replace traditional `for` loops and `append()` calls, making your code easier to read and often faster to execute.

**Example:**
`squares = [x**2 for x in range(5)]`  # Result: [0, 1, 4, 9, 16]

--- Parallel Output ---
POEM:
 A spark of code within a silent frame,
It learns the world and whispers back its name.
JOKE:
 I asked my AI to tell me a joke, but it said it didn't want to spoil the punchline before it had finished learning my sense of humor.


--- 
## 5. Memory & LangGraph's Built-in Persistence (`MemorySaver`)

In modern LangChain, **LangGraph's `MemorySaver` checkpointer** is the official standard for multi-turn state persistence.

It persists chat histories per `thread_id` without deprecation warnings.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END, MessagesState

# Define graph node to process messages
def call_model(state: MessagesState):
    response = llm.invoke(state['messages'])
    return {'messages': [response]}

# Build StateGraph
workflow = StateGraph(MessagesState)
workflow.add_node('model', call_model)
workflow.add_edge(START, 'model')
workflow.add_edge('model', END)

# Attach MemorySaver checkpointer for state persistence
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

# Thread Configuration
config = {'configurable': {'thread_id': 'conversation_1'}}

# Turn 1
res1 = app.invoke({'messages': [('user', 'Hi, my name is Alice and I live in San Francisco.')]}, config=config)
print("Turn 1 Response:", res1['messages'][-1].content)

# Turn 2 (Testing Memory Persistence across thread_id)
res2 = app.invoke({'messages': [('user', 'What is my name and where do I live?')]}, config=config)
print("\nTurn 2 Response (Memory Check):")
print(res2['messages'][-1].content)

--- 
## 6. Understanding Tool Calling: LLM Request vs. Tool Execution

### 💡 How Tool Calling Works (The 3-Step Lifecycle):
1. **Step 1: LLM Tool Request**: When you call `llm_with_tools.invoke("What is 35 multiplied by 12?")`, the LLM **does not run code**. Instead, it generates a **structured JSON request** specifying which tool to run and with what arguments.
2. **Step 2: Python Tool Execution**: Your application receives the `tool_calls` dictionary, executes the actual Python function (`multiply_numbers(a=35, b=12)`), and gets the output (`420.0`).
3. **Step 3: Final LLM Answer**: You send the tool output back to the LLM as a `ToolMessage`, allowing the LLM to format the final human answer.

In [11]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

# 1. Define custom tools using @tool decorator
@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b

@tool
def multiply_numbers(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

tools = [add_numbers, multiply_numbers]

# 2. Bind tools to local LLM
llm_with_tools = llm.bind_tools(tools)

# --- STEP 1: LLM generates a Tool Call Request ---
user_query = "What is 35 multiplied by 12?"
messages = [HumanMessage(content=user_query)]

ai_tool_request = llm_with_tools.invoke(messages)
print("=== STEP 1: LLM Tool Call Request ===")
print("Model Output Type:", type(ai_tool_request))
print("Tool Calls Requested by LLM:", ai_tool_request.tool_calls)

# --- STEP 2: Execute the Python Tool Function ---
tool_call = ai_tool_request.tool_calls[0]
print("\n=== STEP 2: Executing Python Function ===")
print(f"Tool Requested: '{tool_call['name']}' with arguments: {tool_call['args']}")

# Run actual Python tool
tool_result = multiply_numbers.invoke(tool_call['args'])
print(f"Python Tool Execution Result: {tool_result}")

# --- STEP 3: Return ToolMessage to LLM for Final Answer ---
messages.append(ai_tool_request)
messages.append(ToolMessage(content=str(tool_result), tool_call_id=tool_call['id']))

final_answer = llm.invoke(messages)
print("\n=== STEP 3: Final LLM Answer to User ===")
print(final_answer.content)

=== STEP 1: LLM Tool Call Request ===
Model Output Type: <class 'langchain_core.messages.ai.AIMessage'>
Tool Calls Requested by LLM: [{'name': 'multiply_numbers', 'args': {'b': 12, 'a': 35}, 'id': 'iJw2LVc1', 'type': 'tool_call'}]

=== STEP 2: Executing Python Function ===
Tool Requested: 'multiply_numbers' with arguments: {'b': 12, 'a': 35}
Python Tool Execution Result: 420.0

=== STEP 3: Final LLM Answer to User ===
[{'type': 'text', 'text': '35 multiplied by 12 is 420.', 'extras': {'signature': 'EjQKMgERTTIPoABX+pWpkZBSmzuQ0Qk8wmjtPvuhSMsnxNIvw/mFsE9BRBZiHqcQ8Wxx1muZ'}}]


--- 
## 7. Building an Interactive Tool-Calling Agent Loop

An **Agent** automates the 3-step loop above: it inspects the user query, dynamically decides which tool(s) to call, runs them, and loops until it produces the final answer.

In [12]:
def run_simple_agent(query: str):
    print(f"User Query: '{query}'")
    messages = [HumanMessage(content=query)]
    
    # Step 1: LLM decides tool call
    ai_msg = llm_with_tools.invoke(messages)
    messages.append(ai_msg)
    
    if ai_msg.tool_calls:
        for tool_call in ai_msg.tool_calls:
            tool_name = tool_call['name']
            tool_args = tool_call['args']
            print(f"🤖 Executing Tool: {tool_name}({tool_args})")
            
            # Match tool name and execute
            if tool_name == "add_numbers":
                result = add_numbers.invoke(tool_args)
            elif tool_name == "multiply_numbers":
                result = multiply_numbers.invoke(tool_args)
            else:
                result = "Tool not found"
                
            # Append ToolMessage with result
            messages.append(ToolMessage(content=str(result), tool_call_id=tool_call['id']))
            
        # Step 2: Final response generation
        final_response = llm.invoke(messages)
        print("Final Answer:", final_response.content)
    else:
        print("Direct Answer:", ai_msg.content)

run_simple_agent("What is 150 multiplied by 4?")

User Query: 'What is 150 multiplied by 4?'
🤖 Executing Tool: multiply_numbers({'a': 150, 'b': 4})
Final Answer: [{'type': 'text', 'text': '150 multiplied by 4 is 600.', 'extras': {'signature': 'EjQKMgERTTIPWoIFYJWUXHXycirun/kaKts2O6RnOIAm17Al4pDtrfYn3k0kq44QLio34Rpj'}}]


--- 
## 8. Retrieval-Augmented Generation (RAG)

RAG provides domain-specific knowledge to your LLM by retrieving context from external documents.

Steps in RAG:
1. **Load & Chunk Text**: Split documents into chunks (`RecursiveCharacterTextSplitter`).
2. **Vector Embeddings**: Compute embeddings with local `OllamaEmbeddings` (`nomic-embed-text:latest`).
3. **Store in Retriever**: Store chunks in a vector store.
4. **LCEL RAG Chain**: Pass retrieved context to the LLM.

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

# Sample Document Knowledge Base
raw_text = """
The GenAI Project Architecture Guide:
1. Local Model Engine: Powered by Ollama running gemma4:e2b and nemotron-3-nano:4b.
2. Vector Search Engine: Utilizes nomic-embed-text for fast 768-dimensional local text embeddings.
3. Agent Framework: Uses LangChain & LangGraph for orchestration, supporting tools and multi-turn memory.
4. Security Protocol: All data remains local with zero cloud API dependency.
"""

# 1. Split Text into Chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)
docs = text_splitter.create_documents([raw_text])
print(f"Split document into {len(docs)} chunks.")

# 2. Local Vector Embeddings & Vector Store
embeddings = OllamaEmbeddings(model=EMBED_MODEL)
vectorstore = InMemoryVectorStore.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 3. Construct LCEL RAG Chain
rag_prompt = ChatPromptTemplate.from_template("""
Answer the question strictly based on the following context:

Context:
{context}

Question: {question}
Answer:
""")

def format_docs(documents):
    return "\n\n".join(doc.page_content for doc in documents)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Query RAG Chain
rag_result = rag_chain.invoke("What is the vector search engine and embedding dimension used in the project?")
print("\n--- RAG Answer ---")
print(rag_result)

Split document into 4 chunks.

--- RAG Answer ---
Answer: The vector search engine is nomic-embed-text, and the embedding dimension is 768.


--- 
## 9. Summary & Next Steps

🎉 **Congratulations!** You have covered all foundational concepts of **LangChain**:
- Model invocation (`invoke`, `stream`, `batch`)
- `ChatPromptTemplate` and message structures
- Output parsing & Pydantic structured output
- LCEL pipelines (`|`, `RunnableParallel`, `RunnablePassthrough`)
- Memory management with LangGraph `MemorySaver` checkpointer
- `@tool` integration and agent loops
- Complete local RAG pipeline with `OllamaEmbeddings` and `InMemoryVectorStore`

### ➡️ Next Tutorial Steps:
- **LangGraph 101**: Stateful, cyclic graph agents and human-in-the-loop workflows.
- **AutoGen 101**: Multi-agent conversational patterns with local Ollama models.